In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 0 ns (started: 2026-03-26 01:15:27 +05:30)


In [2]:
import tensorflow as tf
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import random

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import layers, models, regularizers
from sklearn.model_selection import StratifiedKFold

from sklearn.utils.class_weight import compute_class_weight

time: 9.59 s (started: 2026-03-26 01:15:27 +05:30)


In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
time: 15 ms (started: 2026-03-26 01:15:37 +05:30)


In [4]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 16 ms (started: 2026-03-26 01:15:37 +05:30)


In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"
SEED = 42

time: 16 ms (started: 2026-03-26 01:15:37 +05:30)


In [6]:
CLASSES = ["NORMAL", "PNEUMONIA"]

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [7]:
bad_files = (
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpeg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.jpg")) +
    list((TRAIN_DIR / "NORMAL").glob("aug_*.png"))
)

deleted = 0

for f in bad_files:
    try:
        f.unlink()
        deleted += 1
    except Exception as e:
        print(f"Error deleting {f}: {e}")

print(f"Deleted {deleted} augmented files")

Deleted 0 augmented files
time: 31 ms (started: 2026-03-26 01:15:37 +05:30)


In [8]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [9]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [10]:
def make_balanced_fold_gen(paths, labels, batch_size=32, datagen=None):
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]

    while True:
        batch_paths, batch_labels = [], []

        for _ in range(batch_size // 2):
            p = np.random.choice(pos_idx)
            n = np.random.choice(neg_idx)

            batch_paths += [paths[p], paths[n]]
            batch_labels += [1, 0]

        combined = list(zip(batch_paths, batch_labels))
        random.shuffle(combined)
        batch_paths, batch_labels = zip(*combined)

        images = []
        for path in batch_paths:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0
        
            img = np.expand_dims(img, axis=-1)  
        
            if datagen is not None:
                img = datagen.random_transform(img)  
        
            images.append(img)

        yield np.array(images), np.array(batch_labels)

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [11]:
label_to_classname = {0: "NORMAL", 1: "PNEUMONIA"}

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [12]:
def make_val_generator(val_paths, val_labels):

    val_df = pd.DataFrame({
        "filename": val_paths,
        "class": [label_to_classname[l] for l in val_labels]
    })

    return val_test_datagen.flow_from_dataframe(
        val_df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        color_mode=COLOR_MODE,
        shuffle=False
    )

time: 16 ms (started: 2026-03-26 01:15:37 +05:30)


In [13]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        layers.Conv2D(32, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.1),

        layers.Conv2D(64, (3,3), padding='same',
                      kernel_regularizer=regularizers.l2(0.0001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),

        layers.Dense(1, activation='sigmoid')
    ])

    m.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

    return m

time: 0 ns (started: 2026-03-26 01:15:37 +05:30)


In [14]:
VALID_EXTS = {".jpeg", ".jpg", ".png"}
 
all_paths = []
all_labels = []

temp_gen = train_datagen.flow_from_directory( #dummy generator
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=1,
    class_mode="binary",
    shuffle=False
)

class_map = temp_gen.class_indices  
 
for class_name, label in class_map.items():
    class_dir = TRAIN_DIR / class_name
 
    for img_name in os.listdir(class_dir):
        if Path(img_name).suffix.lower() not in VALID_EXTS:
            continue
        all_paths.append(str(class_dir / img_name))
        all_labels.append(label)
 
all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
 
print("Total samples:", len(all_paths))
print("Class distribution:", np.bincount(all_labels))

Found 4679 images belonging to 2 classes.
Total samples: 4679
Class distribution: [1199 3480]
time: 578 ms (started: 2026-03-26 01:15:37 +05:30)


In [15]:
skf = StratifiedKFold(
    n_splits=3,  
    shuffle=True,
    random_state=42
)

time: 0 ns (started: 2026-03-26 01:15:38 +05:30)


In [16]:
for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels), 1):

    print(f"\n------------ Fold {fold} ------------")

    train_paths_fold = all_paths[train_idx]
    val_paths_fold   = all_paths[val_idx]

    train_labels_fold = all_labels[train_idx]
    val_labels_fold   = all_labels[val_idx]

    train_gen = make_balanced_fold_gen(
        train_paths_fold,
        train_labels_fold,
        batch_size=BATCH_SIZE,
        datagen=train_datagen
    )

    val_gen = make_val_generator(val_paths_fold, val_labels_fold)

    model = build_model()

    steps_per_epoch = len(train_paths_fold) // BATCH_SIZE

    callbacks = [
        EarlyStopping(patience=5, restore_best_weights=True),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=2,
            min_lr=1e-6
        ),
        ModelCheckpoint(
            filepath=f"best_model_fold_{fold}.h5",
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]

    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        epochs=15,
        callbacks=callbacks
    )



------------ Fold 1 ------------
Found 1560 validated image filenames belonging to 2 classes.
Epoch 1/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 980ms/step - accuracy: 0.6347 - loss: 2.9132 - precision: 0.6393 - recall: 0.6148
Epoch 1: val_accuracy improved from None to 0.74359, saving model to best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.7403 - loss: 1.2494 - precision: 0.7500 - recall: 0.7210 - val_accuracy: 0.7436 - val_loss: 0.5354 - val_precision: 0.7436 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.8704 - loss: 0.3121 - precision: 0.8870 - recall: 0.8491
Epoch 2: val_accuracy did not improve from 0.74359
97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8779 - loss: 0.3000 - precision: 0.8928 - recall: 0.8589 - val_accuracy: 0.7436 - val_loss: 0.7285 - val_precision: 0.7436 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 941ms/step - accuracy: 0.8965 - loss: 0.2716 - precision: 0.9141 - recall: 0.8753
Epoch 3: val_accuracy did not improve from 0.74359
97/97 ━━━━━━━━━━━━━━━━━━━━ 105s 1s/step - accuracy: 0.9034 - loss: 0.2564 - precision: 0.9179 - recall: 0.8860 - val_accuracy: 0.7436 - val_loss: 1.1106 - val_precision: 0.7436 - val_recall: 1.0000 - 

97/97 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.9291 - loss: 0.2029 - precision: 0.9370 - recall: 0.9201 - val_accuracy: 0.7526 - val_loss: 0.6102 - val_precision: 0.7503 - val_recall: 1.0000 - learning_rate: 3.0000e-05
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 805ms/step - accuracy: 0.9258 - loss: 0.2015 - precision: 0.9398 - recall: 0.9100
Epoch 6: val_accuracy improved from 0.75256 to 0.90385, saving model to best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 93s 964ms/step - accuracy: 0.9262 - loss: 0.2002 - precision: 0.9361 - recall: 0.9149 - val_accuracy: 0.9038 - val_loss: 0.2244 - val_precision: 0.8909 - val_recall: 0.9922 - learning_rate: 9.0000e-06
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 808ms/step - accuracy: 0.9289 - loss: 0.2050 - precision: 0.9447 - recall: 0.9113
Epoch 7: val_accuracy improved from 0.90385 to 0.94231, saving model to best_model_fold_1.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 968ms/step - accuracy: 0.9294 - loss: 0.2045 - precision: 0.9446 - recall: 0.9124 - val_accuracy: 0.9423 - val_loss: 0.1486 - val_precision: 0.9837 - val_recall: 0.9379 - learning_rate: 9.0000e-06
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 820ms/step - accuracy: 0.9238 - loss: 0.2010 - precision: 0.9394 - recall: 0.9061
Epoch 8: val_accuracy did not improve from 0.94231
97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 973ms/step - accuracy: 0.9269 - loss: 0.2008 - precision: 0.9414 - recall: 0.9104 - val_accuracy: 0.8872 - val_loss: 0.2832 - val_precision: 0.9940 - val_recall: 0.8534 - learning_rate: 9.0000e-06
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 817ms/step - accuracy: 0.9304 - loss: 0.1872 - precision: 0.9481 - recall: 0.9106
Epoch 9: val_accuracy did not improve from 0.94231
97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 973ms/step - accuracy: 0.9301 - loss: 0.1964 - precision: 0.9459 - recall: 0.9124 - val_accuracy: 0.8801 - val_loss: 0.3188 - val_precision: 0.9949 - val_recall: 0.8

...

97/97 ━━━━━━━━━━━━━━━━━━━━ 96s 989ms/step - accuracy: 0.9243 - loss: 0.2102 - precision_1: 0.9358 - recall_1: 0.9111 - val_accuracy: 0.9449 - val_loss: 0.1465 - val_precision_1: 0.9847 - val_recall_1: 0.9405 - learning_rate: 3.0000e-05
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 830ms/step - accuracy: 0.9273 - loss: 0.2194 - precision_1: 0.9298 - recall_1: 0.9242
Epoch 8: val_accuracy did not improve from 0.94487
97/97 ━━━━━━━━━━━━━━━━━━━━ 95s 983ms/step - accuracy: 0.9227 - loss: 0.2209 - precision_1: 0.9276 - recall_1: 0.9169 - val_accuracy: 0.9006 - val_loss: 0.2585 - val_precision_1: 0.9951 - val_recall_1: 0.8707 - learning_rate: 3.0000e-05
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 836ms/step - accuracy: 0.9341 - loss: 0.1933 - precision_1: 0.9381 - recall_1: 0.9298
Epoch 9: val_accuracy did not improve from 0.94487
97/97 ━━━━━━━━━━━━━━━━━━━━ 96s 991ms/step - accuracy: 0.9317 - loss: 0.1890 - precision_1: 0.9385 - recall_1: 0.9240 - val_accuracy: 0.8859 - val_loss: 0.3059 - val_precisi

97/97 ━━━━━━━━━━━━━━━━━━━━ 110s 1s/step - accuracy: 0.7729 - loss: 0.9995 - precision_2: 0.7759 - recall_2: 0.7674 - val_accuracy: 0.7441 - val_loss: 0.5186 - val_precision_2: 0.7441 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 926ms/step - accuracy: 0.8374 - loss: 0.3529 - precision_2: 0.8410 - recall_2: 0.8327
Epoch 2: val_accuracy did not improve from 0.74407
97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8512 - loss: 0.3384 - precision_2: 0.8534 - recall_2: 0.8479 - val_accuracy: 0.7441 - val_loss: 0.6413 - val_precision_2: 0.7441 - val_recall_2: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 926ms/step - accuracy: 0.8739 - loss: 0.3050 - precision_2: 0.8800 - recall_2: 0.8660
Epoch 3: val_accuracy did not improve from 0.74407
97/97 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.8705 - loss: 0.3112 - precision_2: 0.8773 - recall_2: 0.8615 - val_accuracy: 0.7441 - val_loss: 0.6726 - val_precision_2: 

97/97 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.9024 - loss: 0.2591 - precision_2: 0.9100 - recall_2: 0.8930 - val_accuracy: 0.8024 - val_loss: 0.3698 - val_precision_2: 0.7902 - val_recall_2: 1.0000 - learning_rate: 3.0000e-05
Epoch 6/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 802ms/step - accuracy: 0.9179 - loss: 0.2413 - precision_2: 0.9248 - recall_2: 0.9098
Epoch 6: val_accuracy improved from 0.80244 to 0.91533, saving model to best_model_fold_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 967ms/step - accuracy: 0.9114 - loss: 0.2415 - precision_2: 0.9176 - recall_2: 0.9040 - val_accuracy: 0.9153 - val_loss: 0.1975 - val_precision_2: 0.9035 - val_recall_2: 0.9922 - learning_rate: 3.0000e-05
Epoch 7/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 812ms/step - accuracy: 0.9297 - loss: 0.2064 - precision_2: 0.9475 - recall_2: 0.9100
Epoch 7: val_accuracy improved from 0.91533 to 0.94997, saving model to best_model_fold_3.h5


97/97 ━━━━━━━━━━━━━━━━━━━━ 94s 972ms/step - accuracy: 0.9301 - loss: 0.2058 - precision_2: 0.9400 - recall_2: 0.9188 - val_accuracy: 0.9500 - val_loss: 0.1318 - val_precision_2: 0.9713 - val_recall_2: 0.9612 - learning_rate: 3.0000e-05
Epoch 8/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 802ms/step - accuracy: 0.9097 - loss: 0.2518 - precision_2: 0.9197 - recall_2: 0.8978
Epoch 8: val_accuracy did not improve from 0.94997
97/97 ━━━━━━━━━━━━━━━━━━━━ 92s 955ms/step - accuracy: 0.9111 - loss: 0.2428 - precision_2: 0.9220 - recall_2: 0.8982 - val_accuracy: 0.9391 - val_loss: 0.1529 - val_precision_2: 0.9863 - val_recall_2: 0.9310 - learning_rate: 3.0000e-05
Epoch 9/15
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.9214 - loss: 0.2009 - precision_2: 0.9249 - recall_2: 0.9173
Epoch 9: val_accuracy did not improve from 0.94997
97/97 ━━━━━━━━━━━━━━━━━━━━ 92s 948ms/step - accuracy: 0.9243 - loss: 0.2020 - precision_2: 0.9290 - recall_2: 0.9188 - val_accuracy: 0.9217 - val_loss: 0.1948 - val_precisi

In [17]:
print("end")

end
time: 0 ns (started: 2026-03-26 02:14:52 +05:30)


In [18]:
#changed callback to val_accuracy

time: 0 ns (started: 2026-03-26 02:14:52 +05:30)
